# Study Overview and EDA Evidence

Lending Club Data contains 2,925,492 originated loans. This notebook reads the published aggregate evidence. The research scripts reconstruct those aggregates from the supplied source. Model probabilities describe recorded contract outcomes among eligible originated loans.

In [1]:
from pathlib import Path
import pandas as pd

repo = Path.cwd()
if not (repo / "models").exists():
    repo = repo.parent

evidence = repo / "research/consumer_credit_application_scorecard_eda_annual_20261108/evidence"
status = pd.read_csv(evidence / "status.csv")
status

status,n,share
Fully Paid,1497783,0.511976
Charged Off,362548,0.123927
Does not meet the credit policy. Status:Fully Paid,1988,0.000680
Does not meet the credit policy. Status:Charged Off,761,0.000260
Current,1031016,0.352425
In Grace Period,10028,0.003428
Late (31-120 days),16154,0.005522
Late (16-30 days),2719,0.000929
Default,433,0.000148
Issued,2062,0.000705


## Annual coverage

The origination-year profiles help assess which loans have recorded outcomes and how their distribution changes.

In [2]:
annual = pd.read_csv(evidence / "annual_outcomes.csv")
annual

year,n,good,bad,unresolved,legacy,labelled_share,term60_share
2007,603,206,45,0,352,0.416252,0.000000
2008,2393,1315,247,0,831,0.652737,0.000000
2009,5281,4122,594,0,565,0.893013,0.000000
2010,12537,10049,1487,0,1001,0.920156,0.269682
2011,21721,18424,3297,0,0,1.000000,0.350813
2012,53367,44723,8644,0,0,1.000000,0.185452
2013,134814,113787,21027,0,0,1.000000,0.255107
2014,235629,193871,41748,10,0,0.999958,0.310059
2015,421095,323125,79712,18258,0,0.956642,0.327532
2016,434407,324022,79082,31303,0,0.927941,0.255318


## Numeric distributions

These summaries use available values across the full file. The Word paper contains the histograms and rank-correlation matrix.

In [3]:
summary = pd.read_csv(evidence / "numeric_summary.csv")
summary

field,n,missing,mean,std,min,p1,q1,median,q3,p99,max,zeros,negative
loan_amnt,2925492,0,15358.775105,9478.383451,500.0,1500.00,8000.00,13000.0,20000.00,40000.00,40000.0,0,0
annual_inc,2925488,4,79937.274015,111747.608710,0.0,16000.00,47000.00,66000.0,95000.00,280000.00,110000000.0,3054,0
dti,2922384,3108,19.296480,15.719773,-1.0,1.75,12.08,18.1,24.88,48.14,999.0,2353,2
fico_mid,2925492,0,702.511133,33.766092,612.0,662.00,677.00,697.0,722.00,807.00,847.5,0,0
revol_util,2922832,2660,48.986439,24.901211,0.0,0.60,29.80,48.6,68.10,98.00,892.3,18856,0


## Study samples

The model training, validation and later cohorts use individual standard-policy 36- and 60-month loans with recorded Good or Bad outcomes.

In [4]:
splits = pd.read_csv(repo / "research/development_sample_20261108/sample_target_results/split_summary.csv")
splits

split,n,good,bad,bad_share,term_36,term_60
Train,86624,73196,13428,0.155015,66037,20587
Validation,134814,113787,21027,0.155970,100422,34392
OOT,235619,193871,41748,0.177184,162570,73049


## Selected model

The primary scorecard uses five WoE predictors and a separate probability intercept for each term. The full comparison supports the choice in favour of transparent point contributions, alongside recorded calibration limitations.